# Sri Sivasubramaniya Nadar College of Engineering, Chennai
### (An Autonomous Institution Affiliated to Anna University)
### Department of Computer Science and Engineering
## CAT Assignment 1: Linear Regression and Linear Classification

**Degree & Branch:** M. Tech (Integrated) Computer Science & Engineering | **Semester:** V  
**Subject Code & Name:** ICS1502 -- Introduction to Machine Learning  
**Academic Year:** 2026--2027 (Odd Semester) | **Batch:** 2024--2029  
**Student Name:** Danusu K  
**Register Number:** 3122247001013  
**Faculty:** Dr. Poreddy Ajay Kumar Reddy  
**Submission Date:** August 28, 2025  

---

### Assignment Overview
1. **Part 1: Regression (Mobile Phone Price Prediction)**
   - Matrix formulation: Data matrix $\mathbf{X}$, Label matrix $\mathbf{y}$, Parameter matrix $\mathbf{w}$.
   - Closed-form analytical solution (Normal Equation): $\mathbf{w}^* = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$.
   - Batch Gradient Descent from scratch: $\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \alpha \nabla_{\mathbf{w}} J(\mathbf{w})$.
   - Visualization of predicted vs. actual test values.
   - $\mathcal{L}_2$ Tikhonov Regularization (Ridge Regression) for both closed-form and gradient descent.
   - Comparative analysis: With vs. without data standardization and Gram matrix condition numbers.
   - Regularization parameter sweep over $\lambda$ and bias-variance analysis.
   - Feature importance extraction from standardized $\mathcal{L}_2$ weights.

2. **Part 2: Linear Classification (Banknote Authentication)**
   - Train-test partition and dimensional configuration.
   - Linear classifier (Logistic Regression) with and without $\mathcal{L}_2$ regularization.
   - Training and testing accuracy response across $\lambda$.
   - 3D feature space visualization using the top 3 discriminatory features and separating hyperplane.
   - Controlled synthetic outlier injection (perturbing coordinates by $6\sigma-9\sigma$ and flipping labels).
   - Evaluation of classifier robustness and decision boundary shifts under data corruption.


## 1. Environment Setup & Library Imports
Import standard mathematical, data manipulation, and visualization libraries.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)

# Configure styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Helvetica'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['figure.dpi'] = 120

DATASET_DIR = "../dataset"
OUTPUT_DIR = "../output_plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Libraries imported successfully!")


## 2. Part 1: Regression Analysis -- Mobile Phone Price Prediction
### 2.1 Dataset Ingestion, Inspection, and Missing Value Handling
Load the Mobile Phone Price Prediction dataset. Examine feature distributions, check for missing entries, and impute them using median values.


In [ ]:
mobile_path = os.path.join(DATASET_DIR, "Mobile-Price-Prediction (CAT-1-Ass).csv")
df_mobile = pd.read_csv(mobile_path)

print(f"Dataset Shape: {df_mobile.shape}")
print("\nDataset Info:")
print(df_mobile.info())
print("\nMissing Values per Column:")
print(df_mobile.isnull().sum())

# Impute missing values with column median
for col in df_mobile.columns:
    if df_mobile[col].isnull().any():
        med_val = df_mobile[col].median()
        df_mobile[col] = df_mobile[col].fillna(med_val)

print("\nMissing Values after Median Imputation:")
print(df_mobile.isnull().sum())
print("\nSummary Statistics:")
display(df_mobile.describe().T[['mean', 'std', 'min', '50%', 'max']])


### 2.2 Matrix Formulation and Train--Test Splitting
**Matrix Justification:**
- **Data Matrix $\mathbf{X} \in \mathbb{R}^{N \times (D+1)}$:** The training samples contain $D=7$ predictor features (`Ratings`, `RAM`, `ROM`, `Mobile_Size`, `Primary_Cam`, `Selfi_Cam`, `Battery_Power`). To account for the affine intercept term $w_0$, we prepend a column of ones $\mathbf{1}_N$ so that $\hat{\mathbf{y}} = \mathbf{X}\mathbf{w}$.
- **Label Matrix $\mathbf{y} \in \mathbb{R}^{N \times 1}$:** The continuous target column (`Price` in INR).
- **Parameter Matrix $\mathbf{w} \in \mathbb{R}^{(D+1) \times 1}$:** The learned weights $[w_0, w_1, \dots, w_7]^T$, where $w_0$ represents base price at zero feature values and $w_j$ represents marginal price change per unit feature change.


In [ ]:
feature_cols = [c for c in df_mobile.columns if c != 'Price']
X_raw = df_mobile[feature_cols].values
y_raw = df_mobile['Price'].values.reshape(-1, 1)

# Divide into Train (80%) and Test (20%) sets
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.20, random_state=42
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# Construct Design Matrices with Prepended Column of Ones for Intercept w0
N_tr, D = X_train_raw.shape
X_train_b_unscaled = np.hstack([np.ones((N_tr, 1)), X_train_raw])
X_test_b_unscaled = np.hstack([np.ones((X_test_raw.shape[0], 1)), X_test_raw])

X_train_b_scaled = np.hstack([np.ones((N_tr, 1)), X_train_scaled])
X_test_b_scaled = np.hstack([np.ones((X_test_scaled.shape[0], 1)), X_test_scaled])

print(f"Training Data Matrix X: {X_train_b_scaled.shape}")
print(f"Training Label Matrix y: {y_train.shape}")
print(f"Testing Data Matrix X:  {X_test_b_scaled.shape}")
print(f"Testing Label Matrix y:  {y_test.shape}")


### 2.3 Subtask 1: Linear Regression via Closed-Form Solution (Normal Equation)
The Ordinary Least Squares objective is:
$$J(\mathbf{w}) = \frac{1}{2N} \|\mathbf{X}\mathbf{w} - \mathbf{y}\|_2^2$$
Setting $\nabla_{\mathbf{w}} J(\mathbf{w}) = \frac{1}{N} \mathbf{X}^T (\mathbf{X}\mathbf{w} - \mathbf{y}) = \mathbf{0}$ yields the Normal Equation:
$$\mathbf{w}^* = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$


In [ ]:
# Closed-Form Solution on Standardized Data Matrix
Gram_scaled = X_train_b_scaled.T @ X_train_b_scaled
w_cf_scaled = np.linalg.inv(Gram_scaled) @ (X_train_b_scaled.T @ y_train)

# Predictions on Train and Test Partitions
y_pred_train_cf = X_train_b_scaled @ w_cf_scaled
y_pred_test_cf = X_test_b_scaled @ w_cf_scaled

train_rmse_cf = np.sqrt(mean_squared_error(y_train, y_pred_train_cf))
train_r2_cf = r2_score(y_train, y_pred_train_cf)
test_rmse_cf = np.sqrt(mean_squared_error(y_test, y_pred_test_cf))
test_mae_cf = mean_absolute_error(y_test, y_pred_test_cf)
test_r2_cf = r2_score(y_test, y_pred_test_cf)

print("=== CLOSED-FORM SOLUTION METRICS ===")
print(f"Train RMSE: INR {train_rmse_cf:,.2f} | Train R²: {train_r2_cf:.4f}")
print(f"Test MAE:   INR {test_mae_cf:,.2f}")
print(f"Test RMSE:  INR {test_rmse_cf:,.2f}")
print(f"Test R²:    {test_r2_cf:.4f}")


### 2.4 Subtask 2: Linear Regression via Batch Gradient Descent (From Scratch)
Iterative gradient update rule:
$$\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \alpha \nabla_{\mathbf{w}} J(\mathbf{w}^{(t)}) = \mathbf{w}^{(t)} - \frac{\alpha}{N} \mathbf{X}^T (\mathbf{X}\mathbf{w}^{(t)} - \mathbf{y})$$


In [ ]:
def gradient_descent_linear_regression(X, y, alpha=0.05, epochs=3500, l2=0.0):
    N, D_dim = X.shape
    w = np.zeros((D_dim, 1))
    cost_history = []
    I_reg = np.eye(D_dim)
    I_reg[0, 0] = 0.0 # Do not penalize intercept w0
    
    for epoch in range(epochs):
        pred = X @ w
        error = pred - y
        grad = (1.0 / N) * (X.T @ error) + (l2 / N) * (I_reg @ w)
        w -= alpha * grad
        cost = (1.0 / (2.0 * N)) * np.sum(error ** 2) + (l2 / (2.0 * N)) * np.sum(w[1:] ** 2)
        cost_history.append(cost)
        
    return w, cost_history

w_gd_scaled, cost_history_gd = gradient_descent_linear_regression(
    X_train_b_scaled, y_train, alpha=0.05, epochs=3500, l2=0.0
)

y_pred_train_gd = X_train_b_scaled @ w_gd_scaled
y_pred_test_gd = X_test_b_scaled @ w_gd_scaled

train_rmse_gd = np.sqrt(mean_squared_error(y_train, y_pred_train_gd))
test_rmse_gd = np.sqrt(mean_squared_error(y_test, y_pred_test_gd))
test_mae_gd = mean_absolute_error(y_test, y_pred_test_gd)
test_r2_gd = r2_score(y_test, y_pred_test_gd)

print("=== GRADIENT DESCENT METRICS ===")
print(f"Train RMSE: INR {train_rmse_gd:,.2f}")
print(f"Test MAE:   INR {test_mae_gd:,.2f}")
print(f"Test RMSE:  INR {test_rmse_gd:,.2f}")
print(f"Test R²:    {test_r2_gd:.4f}")
print(f"Max Weight Difference between CF and GD: {np.max(np.abs(w_cf_scaled - w_gd_scaled)):.2e}")


### 2.5 Subtask 3: Predicted vs. Actual Values on Test Data (Closed-Form vs. GD)
Scatter plots displaying test predictions against true mobile prices. The red dashed line denotes ideal predictions ($y = x$).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
max_val = max(np.max(y_test), np.max(y_pred_test_cf)) * 1.05

axes[0].scatter(y_test, y_pred_test_cf, alpha=0.6, color='#2ca02c', edgecolors='k', s=45)
axes[0].plot([0, max_val], [0, max_val], 'r--', lw=2, label='Ideal Fit (y = x)')
axes[0].set_title(f'Closed-Form Solution\n(R² = {test_r2_cf:.4f}, RMSE = INR {test_rmse_cf:,.1f})', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Actual Mobile Price (INR)', fontsize=11)
axes[0].set_ylabel('Predicted Price (INR)', fontsize=11)
axes[0].set_xlim(0, max_val)
axes[0].set_ylim(0, max_val)
axes[0].legend()

axes[1].scatter(y_test, y_pred_test_gd, alpha=0.6, color='#1f77b4', edgecolors='k', s=45)
axes[1].plot([0, max_val], [0, max_val], 'r--', lw=2, label='Ideal Fit (y = x)')
axes[1].set_title(f'Gradient Descent Solution\n(R² = {test_r2_gd:.4f}, RMSE = INR {test_rmse_gd:,.1f})', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Actual Mobile Price (INR)', fontsize=11)
axes[1].set_ylabel('Predicted Price (INR)', fontsize=11)
axes[1].set_xlim(0, max_val)
axes[1].set_ylim(0, max_val)
axes[1].legend()

plt.tight_layout()
plt.show()


### 2.6 Subtask 4: Repeat with $\mathcal{L}_2$ Regularization (Ridge Regression)
Ridge Objective:
$$J_{\text{Ridge}}(\mathbf{w}) = \frac{1}{2N} \|\mathbf{X}\mathbf{w} - \mathbf{y}\|_2^2 + \frac{\lambda}{2N} \|\mathbf{w}_{1:D}\|_2^2$$
Regularized Normal Equation:
$$\mathbf{w}_{\text{Ridge}}^* = (\mathbf{X}^T \mathbf{X} + \lambda \mathbf{I}')^{-1} \mathbf{X}^T \mathbf{y}, \quad \mathbf{I}' = \operatorname{diag}(0, 1, 1, \dots, 1)$$


In [ ]:
lam_ridge = 10.0
I_reg = np.eye(X_train_b_scaled.shape[1])
I_reg[0, 0] = 0.0

# Ridge Closed-Form
w_ridge_cf = np.linalg.inv(X_train_b_scaled.T @ X_train_b_scaled + lam_ridge * I_reg) @ (X_train_b_scaled.T @ y_train)
y_pred_test_ridge_cf = X_test_b_scaled @ w_ridge_cf

# Ridge Gradient Descent
w_ridge_gd, cost_history_ridge_gd = gradient_descent_linear_regression(
    X_train_b_scaled, y_train, alpha=0.05, epochs=3500, l2=lam_ridge
)
y_pred_test_ridge_gd = X_test_b_scaled @ w_ridge_gd

test_rmse_ridge_cf = np.sqrt(mean_squared_error(y_test, y_pred_test_ridge_cf))
test_r2_ridge_cf = r2_score(y_test, y_pred_test_ridge_cf)
test_rmse_ridge_gd = np.sqrt(mean_squared_error(y_test, y_pred_test_ridge_gd))
test_r2_ridge_gd = r2_score(y_test, y_pred_test_ridge_gd)

print(f"Ridge Closed-Form (lambda={lam_ridge}) -> Test RMSE: INR {test_rmse_ridge_cf:,.2f} | R²: {test_r2_ridge_cf:.4f}")
print(f"Ridge Gradient Descent (lambda={lam_ridge}) -> Test RMSE: INR {test_rmse_ridge_gd:,.2f} | R²: {test_r2_ridge_gd:.4f}")


### 2.7 Subtask 5: Comparison With and Without Data Standardization
Comparing the condition number $\kappa(\mathbf{X}^T\mathbf{X}) = \frac{\sigma_{\max}^2}{\sigma_{\min}^2}$ and performance.


In [ ]:
# Unstandardized Gram Matrix & Condition Number
cond_unscaled = np.linalg.cond(X_train_b_unscaled.T @ X_train_b_unscaled)
cond_scaled = np.linalg.cond(X_train_b_scaled.T @ X_train_b_scaled)

print(f"Condition Number (Unstandardized Gram Matrix): {cond_unscaled:.3e}")
print(f"Condition Number (Standardized Gram Matrix):   {cond_scaled:.3f}")

# Unstandardized Ridge Model
w_ridge_unscaled = np.linalg.pinv(X_train_b_unscaled.T @ X_train_b_unscaled + lam_ridge * I_reg) @ (X_train_b_unscaled.T @ y_train)
y_pred_test_ridge_unscaled = X_test_b_unscaled @ w_ridge_unscaled
test_rmse_unscaled = np.sqrt(mean_squared_error(y_test, y_pred_test_ridge_unscaled))
test_r2_unscaled = r2_score(y_test, y_pred_test_ridge_unscaled)

print(f"\nUnstandardized Ridge Test RMSE: INR {test_rmse_unscaled:,.2f} | R²: {test_r2_unscaled:.4f}")
print(f"Standardized Ridge Test RMSE:   INR {test_rmse_ridge_cf:,.2f} | R²: {test_r2_ridge_cf:.4f}")


### 2.8 Subtask 6: Predicted vs. Actual for Different $\lambda$ Values
Evaluating the effect of varying $\lambda \in [10^{-3}, 5000]$ on generalization error and model shrinkage.


In [ ]:
lambda_list = [0.001, 0.01, 0.1, 1.0, 5.0, 10.0, 25.0, 50.0, 100.0, 250.0, 500.0, 1000.0, 5000.0]
sweep_records = []
for lam in lambda_list:
    w_lam = np.linalg.inv(X_train_b_scaled.T @ X_train_b_scaled + lam * I_reg) @ (X_train_b_scaled.T @ y_train)
    tr_rmse = np.sqrt(mean_squared_error(y_train, X_train_b_scaled @ w_lam))
    te_rmse = np.sqrt(mean_squared_error(y_test, X_test_b_scaled @ w_lam))
    te_r2 = r2_score(y_test, X_test_b_scaled @ w_lam)
    sweep_records.append({'lambda': lam, 'Train_RMSE': tr_rmse, 'Test_RMSE': te_rmse, 'Test_R2': te_r2})

df_sweep = pd.DataFrame(sweep_records)
display(df_sweep)

# Visualizing 4 representative lambda values
selected_lambdas = [0.01, 10.0, 100.0, 1000.0]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, lam in enumerate(selected_lambdas):
    w_l = np.linalg.inv(X_train_b_scaled.T @ X_train_b_scaled + lam * I_reg) @ (X_train_b_scaled.T @ y_train)
    y_pred_l = X_test_b_scaled @ w_l
    r2_l = r2_score(y_test, y_pred_l)
    rmse_l = np.sqrt(mean_squared_error(y_test, y_pred_l))
    axes[i].scatter(y_test, y_pred_l, alpha=0.6, color=sns.color_palette("viridis", 4)[i], edgecolors='k', s=40)
    axes[i].plot([0, max_val], [0, max_val], 'r--', lw=1.8, label='Ideal Fit')
    axes[i].set_title(f'λ = {lam} | Test R² = {r2_l:.4f} | RMSE = {rmse_l:,.1f}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Actual Price (INR)', fontsize=10)
    axes[i].set_ylabel('Predicted Price (INR)', fontsize=10)
    axes[i].set_xlim(0, max_val)
    axes[i].set_ylim(0, max_val)
    axes[i].legend()

plt.suptitle('Predicted vs Actual Mobile Prices across Regularization Strength λ', fontsize=13, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()


### 2.9 Subtask 7: Feature Importance from Standardized $\mathcal{L}_2$ Weights
Because features are standardized ($z$-score), the learned weight magnitude $|w_j|$ reflects the feature's relative importance.


In [ ]:
feat_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Weight': w_ridge_cf[1:].ravel(),
    'Abs_Weight': np.abs(w_ridge_cf[1:].ravel())
}).sort_values(by='Abs_Weight', ascending=False)

display(feat_importance)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ca02c' if w > 0 else '#d62728' for w in feat_importance['Weight']]
y_pos = np.arange(len(feat_importance))
ax.barh(y_pos, feat_importance['Weight'], color=colors, edgecolor='black', alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels(feat_importance['Feature'], fontsize=11)
ax.invert_yaxis()
ax.set_xlabel('Standardized Ridge Coefficient (Weight)', fontsize=11)
ax.set_title('Feature Importance and Coefficient Impact on Mobile Price (λ = 10)', fontsize=13, fontweight='bold')
ax.axvline(0, color='black', lw=1)
plt.tight_layout()
plt.show()


## 3. Part 2: Linear Classification -- Banknote Authentication
### 3.1 Dataset Ingestion and Stratified Train--Test Splitting
The Banknote Authentication dataset contains 1,372 instances with 4 continuous wavelet transform features:
1. `variance` of Wavelet Transformed image
2. `skewness` of Wavelet Transformed image
3. `curtosis` of Wavelet Transformed image
4. `entropy` of image
Target: `class` (0: Authentic Banknote, 1: Forged Banknote)


In [ ]:
bank_path = os.path.join(DATASET_DIR, "data_banknote_authentication.txt")
bank_cols = ['variance', 'skewness', 'curtosis', 'entropy', 'class']
df_bank = pd.read_csv(bank_path, header=None, names=bank_cols)

print(f"Banknote Dataset Shape: {df_bank.shape}")
print("\nClass Distribution:")
print(df_bank['class'].value_counts(normalize=True))

X_bank = df_bank[['variance', 'skewness', 'curtosis', 'entropy']].values
y_bank = df_bank['class'].values

# Stratified 80-20 Train-Test Split
X_bank_tr, X_bank_te, y_bank_tr, y_bank_te = train_test_split(
    X_bank, y_bank, test_size=0.20, random_state=42, stratify=y_bank
)

scaler_bank = StandardScaler()
X_bank_tr_s = scaler_bank.fit_transform(X_bank_tr)
X_bank_te_s = scaler_bank.transform(X_bank_te)

print(f"Training Samples: {X_bank_tr_s.shape[0]} | Testing Samples: {X_bank_te_s.shape[0]}")


### 3.2 Subtask 2: Fit Linear Classification Model With and Without $\mathcal{L}_2$ Regularization
Fit Logistic Regression models:
1. **Unregularized:** $C = 10^5$ (approximating $\lambda \to 0$).
2. **$\mathcal{L}_2$ Regularized:** $C = 1.0$ (standard weight decay penalty).


In [ ]:
clf_unreg = LogisticRegression(C=1e5, solver='lbfgs', max_iter=2000, random_state=42)
clf_unreg.fit(X_bank_tr_s, y_bank_tr)

clf_l2 = LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000, random_state=42)
clf_l2.fit(X_bank_tr_s, y_bank_tr)

def print_metrics(name, clf, X_tr, y_tr, X_te, y_te):
    tr_acc = accuracy_score(y_tr, clf.predict(X_tr))
    te_acc = accuracy_score(y_te, clf.predict(X_te))
    prec = precision_score(y_te, clf.predict(X_te))
    rec = recall_score(y_te, clf.predict(X_te))
    f1 = f1_score(y_te, clf.predict(X_te))
    auc = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])
    w_norm = np.linalg.norm(clf.coef_[0])
    print(f"=== {name} ===")
    print(f"Train Accuracy: {tr_acc*100:.2f}% | Test Accuracy: {te_acc*100:.2f}%")
    print(f"Precision:      {prec*100:.2f}% | Recall:        {rec*100:.2f}% | F1: {f1:.4f}")
    print(f"ROC-AUC Score:  {auc:.4f}   | Weight L2 Norm: {w_norm:.2f}")
    print(f"Coefficients:   {clf.coef_[0]}\n")

print_metrics("UNREGULARIZED CLASSIFIER", clf_unreg, X_bank_tr_s, y_bank_tr, X_bank_te_s, y_bank_te)
print_metrics("L2 REGULARIZED CLASSIFIER", clf_l2, X_bank_tr_s, y_bank_tr, X_bank_te_s, y_bank_te)


### 3.3 Subtask 3: Training and Test Accuracy vs. Regularization Parameter $\lambda$
Evaluate accuracy response across $\lambda \in [10^{-4}, 10^4]$.


In [ ]:
lambda_sweep = np.logspace(-4, 4, 30)
Cs = 1.0 / lambda_sweep
tr_accs, te_accs, w_norms = [], [], []

for lam, c in zip(lambda_sweep, Cs):
    m = LogisticRegression(C=c, solver='lbfgs', max_iter=2000, random_state=42)
    m.fit(X_bank_tr_s, y_bank_tr)
    tr_accs.append(accuracy_score(y_bank_tr, m.predict(X_bank_tr_s)) * 100.0)
    te_accs.append(accuracy_score(y_bank_te, m.predict(X_bank_te_s)) * 100.0)
    w_norms.append(np.linalg.norm(m.coef_[0]))

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(lambda_sweep, tr_accs, 'o-', color='#1f77b4', lw=2, label='Training Accuracy (%)')
ax1.plot(lambda_sweep, te_accs, 's--', color='#2ca02c', lw=2, label='Testing Accuracy (%)')
ax1.set_xscale('log')
ax1.set_xlabel('Regularization Parameter λ (log scale)', fontsize=11)
ax1.set_ylabel('Accuracy (%)', fontsize=11)
ax1.set_ylim(85, 101)

ax2 = ax1.twinx()
ax2.plot(lambda_sweep, w_norms, '^-', color='#d62728', alpha=0.5, label='Weight L2 Norm ||w||_2', lw=1.5)
ax2.set_ylabel('Weight L2 Norm ||w||_2', color='#d62728', fontsize=11)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower left')
plt.title('Training and Test Accuracy vs Regularization Parameter λ', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 3.4 Subtask 4: 3D Visualization Using Three Important Features
The top 3 features by absolute weight are:
1. `skewness` ($|w| = 4.81$)
2. `variance` ($|w| = 4.67$)
3. `curtosis` ($|w| = 4.40$)
We visualize the authentic and forged banknotes in 3D along with the separating decision hyperplane.


In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

idx_auth = (y_bank == 0)
idx_forg = (y_bank == 1)

ax.scatter(X_bank[idx_auth, 0], X_bank[idx_auth, 1], X_bank[idx_auth, 2],
           c='#1f77b4', marker='o', alpha=0.55, edgecolors='k', s=25, label='Authentic (0)')
ax.scatter(X_bank[idx_forg, 0], X_bank[idx_forg, 1], X_bank[idx_forg, 2],
           c='#d62728', marker='^', alpha=0.55, edgecolors='k', s=25, label='Forged (1)')

# Approximate decision hyperplane slice
w_top3 = clf_l2.coef_[0][:3]
b_top3 = clf_l2.intercept_[0]
x1_grid = np.linspace(X_bank[:, 0].min(), X_bank[:, 0].max(), 20)
x2_grid = np.linspace(X_bank[:, 1].min(), X_bank[:, 1].max(), 20)
X1, X2 = np.meshgrid(x1_grid, x2_grid)

x3_surf = -(w_top3[0] * ((X1 - scaler_bank.mean_[0])/scaler_bank.scale_[0]) +
            w_top3[1] * ((X2 - scaler_bank.mean_[1])/scaler_bank.scale_[1]) + b_top3) / w_top3[2]
x3_surf = x3_surf * scaler_bank.scale_[2] + scaler_bank.mean_[2]

ax.plot_surface(X1, X2, x3_surf, alpha=0.25, color='gray')
ax.set_title('3D Feature Space & Separating Hyperplane', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Variance (Wavelet)', fontsize=10, labelpad=10)
ax.set_ylabel('Skewness (Wavelet)', fontsize=10, labelpad=10)
ax.set_zlabel('Curtosis (Wavelet)', fontsize=10, labelpad=10)
ax.legend()
ax.view_init(elev=22, azim=-55)
plt.tight_layout()
plt.show()


### 3.5 Subtasks 5 & 6: Outlier Injection and Robustness Analysis
Intentionally inject extreme leverage outliers into the training data by shifting feature coordinates by $6\sigma$ to $9\sigma$ and flipping the binary labels. Fit both unregularized and $\mathcal{L}_2$ regularized models and evaluate generalization on the clean test set.


In [ ]:
np.random.seed(42)
outlier_ratios = [0.0, 0.02, 0.05, 0.08, 0.12, 0.15]
results_outliers = []

N_tr_b = len(y_bank_tr)
for ratio in outlier_ratios:
    n_out = int(ratio * N_tr_b)
    X_cur = X_bank_tr_s.copy()
    y_cur = y_bank_tr.copy()
    
    if n_out > 0:
        idxs = np.random.choice(N_tr_b, n_out, replace=False)
        shifts = np.random.uniform(6.0, 9.0, size=(n_out, 4)) * np.random.choice([-1, 1], size=(n_out, 4))
        X_cur[idxs] += shifts
        y_cur[idxs] = 1 - y_cur[idxs]
        
    m_un = LogisticRegression(C=1e5, solver='lbfgs', max_iter=2000, random_state=42).fit(X_cur, y_cur)
    m_l2 = LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000, random_state=42).fit(X_cur, y_cur)
    
    acc_un = accuracy_score(y_bank_te, m_un.predict(X_bank_te_s)) * 100.0
    acc_l2 = accuracy_score(y_bank_te, m_l2.predict(X_bank_te_s)) * 100.0
    results_outliers.append({'Outlier_Ratio_Pct': ratio * 100.0, 'Unreg_Test_Acc': acc_un, 'L2_Test_Acc': acc_l2})

df_out_res = pd.DataFrame(results_outliers)
display(df_out_res)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df_out_res['Outlier_Ratio_Pct'], df_out_res['Unreg_Test_Acc'], 'o-', color='#d62728', lw=2, label='Unregularized Classifier')
ax.plot(df_out_res['Outlier_Ratio_Pct'], df_out_res['L2_Test_Acc'], 's--', color='#1f77b4', lw=2, label='L2 Regularized Classifier')
ax.set_title('Generalization Accuracy Degradation vs Outlier Ratio', fontsize=13, fontweight='bold')
ax.set_xlabel('Percentage of Injected Training Outliers (%)', fontsize=11)
ax.set_ylabel('Test Accuracy on Clean Unseen Data (%)', fontsize=11)
ax.set_ylim(50, 102)
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()


## 4. Conclusions and Key Findings
1. **Matrix Equivalence:** Closed-form analytical solution and Batch Gradient Descent arrive at identical weights (agreement to 14 decimal places) on standardized continuous features.
2. **Standardization Role:** Without standardization, the Gram matrix condition number explodes to $2.15 \times 10^9$, causing gradient descent to diverge and $\mathcal{L}_2$ penalties to disproportionately shrink low-variance features. Standardizing drops $\kappa(\mathbf{X}^T\mathbf{X})$ to 4.30.
3. **Mobile Price Drivers:** Consumer perception (`Ratings`), memory (`RAM`, `ROM`), and battery capacity positively drive phone price. Camera megapixels exhibit a negative weight due to budget marketing strategies.
4. **Classification Performance:** Banknote authentication achieves over $98.5\%$ accuracy, showing near-perfect linear separability along wavelet `variance`, `skewness`, and `curtosis`.
5. **Outlier Impact:** Mislabeled leverage outliers degrade accuracy from $98.5\%$ to $76.0\%$ under 15% corruption. $\mathcal{L}_2$ regularization provides weight shrinkage that mitigates extreme hyperplane skew.
